# Suite MUT — Mutations: the schemas of a mutation log

`Mutations` holds the schemas of a mutation log ([Serialization](../../docs/SERIALIZATION.md), Mutations : Vocabulary):
reference verbs at the top level (`Log`, a list of `Reference`), value verbs nested in them, and the nouns `Ref`, `Def`
and `Slice`. Mutations are schema-governed: every store holds their schemas, and a log is built and written as any value
is. Every store also holds the basic scalars, `Schemas.OfBool` to `Schemas.OfBytes`.

As in Python, with the language's differences (docs/EQUIVALENCE.md): builders and the store's paths are `Proxy`
objects typed loosely, so callbacks are annotated `(c: any)`; ints are `bigint`s, plain mappings are `Map`s (compared
with `equal` against `map(...)`), and `Exists`' noun keeps Python's name, `definition`.

In [ ]:
import { Builders as B, JSON, Plain } from "@mbse/schemas/Framework";
import { assert, equal, map, raises, same } from "./support.js";

const store: any = new B.Store();
const M = { // the store's own mutation schemas, through its paths
  Log: store.Mutations.Log.as_schema(), Reference: store.Mutations.Reference.as_schema(),
  OfAny: store.Mutations.OfAny.as_schema(), OfContainer: store.Mutations.OfContainer.as_schema(),
  OfLink: store.Mutations.OfLink.as_schema(), PropertySet: store.Mutations.OfProperty.Set.as_schema(),
  AnySet: store.Mutations.OfAny.Set.as_schema(), ScalarSet: store.Mutations.OfScalar.Set.as_schema(),
  SCHEMAS: store.names().filter((name: string) => name.startsWith("Mutations.")).map((name: string) => store.registered(name)),
};

/** A `Mutations.OfObject.Ref` naming `key`. */
function ref(key: string): any {
  return store.Mutations.OfObject.Ref((r: any) => r.key(key));
}

/** A `Mutations.Log` of `verbs`, each a `Mutations.Reference` value. */
function log(...verbs: any[]): any {
  return store.Mutations.Log((c: any) => verbs.map((v, i) => c.at(BigInt(i)).as_union(v)));
}

function plain(value: unknown, schema: unknown = M.Log): any {
  return Plain.ToPlain(store)(schema, value);
}

/** The keys of a plain mapping, as Python's `list(mapping)`. */
const keys = (mapping: Map<string, unknown>) => [...mapping.keys()];

## MUT-01 · The mutation schemas, valid, and the basic scalars, held by every store

In [ ]:
{
  assert(M.SCHEMAS.every((schema: any) => schema.validate().length === 0) && M.SCHEMAS.length === 42);
  for (const schema of M.SCHEMAS) {
    assert(store.registered(schema.name) === schema);
  }
  const other: any = new B.Store();
  assert(other.registered("Mutations.Log") !== M.Log && other.Mutations.Log.as_schema().validate().length === 0); // each its own
  for (const name of ["OfBool", "OfInt", "OfFloat", "OfStr", "OfBytes"] as const) {
    const scalar = store.Schemas[name].as_schema();
    assert(scalar.name === `Schemas.${name}` && store.registered(scalar.name) === scalar);
  }
  assert(store.Schemas.OfStr.as_schema().type === String && store.Schemas.OfObject.as_schema().properties.get("ref")?.schema === store.Schemas.OfBool.as_schema());
  assert(same(M.Reference.branches.map((b: any) => b.name), ["exists", "as_property", "insert", "remove", "clear", "as_entries"]));
  assert(same(M.OfAny.branches.map((b: any) => b.name), ["set", "clear", "as_scalar", "as_container", "as_object", "as_union",
    "as_intersection"]));
  assert(M.PropertySet !== M.AnySet && M.AnySet !== M.ScalarSet); // each family its own
}

## MUT-02 · Reference verbs: an object exists, with its properties; a property changed through the nested value verbs

In [ ]:
{
  const exists = store.Mutations.Reference((u: any) => u.exists((e: any) => e.definition((p: any) => p.as_object((d: any) => (
    d
      .key("crm.Person/ada")
      .properties({ name: "Ada", age: 36n })
  )))));
  const aged = store.Mutations.Reference((u: any) => u.as_property((p: any) => (
    p
      .ref(ref("crm.Person/ada"))
      .name("age")
      .mutations([store.Mutations.OfProperty((v: any) => v.set((s: any) => s.value(37n)))])
  )));
  const deep = store.Mutations.Reference((u: any) => u.as_property((p: any) => (
    p
      .ref(ref("crm.Person/ada"))
      .name("address")
      .mutations([store.Mutations.OfProperty((v: any) => v.as_any((a: any) => a.mutations([
        store.Mutations.OfAny((x: any) => x.as_object((o: any) => o.mutations([
          store.Mutations.OfObject.AsProperty((q: any) => q.name("city").mutations([
            store.Mutations.OfProperty((w: any) => w.set((s: any) => s.value("Paris"))),
          ])),
        ]))),
      ])))])
  )));
  const written = plain(log(exists, aged, deep));
  assert(equal(written[0], map({ exists: { definition: { key: "crm.Person/ada",
    properties: { name: { str: "Ada" }, age: { int: 36n } } } } })));
  assert(equal(written[1], map({ as_property: { ref: { key: "crm.Person/ada" }, name: "age",
    mutations: [{ set: { value: { int: 37n } } }] } })));
  const nested = written[2].get("as_property").get("mutations")[0].get("as_any").get("mutations")[0].get("as_object")
    .get("mutations")[0];
  assert(equal(nested, map({ name: "city", mutations: [{ set: { value: { str: "Paris" } } }] }))); // no ref: the path gives it
}

## MUT-03 · Value verbs: containers by slices, by count, and item by item in modulo order; scalars; unions and intersections

In [ ]:
{
  const items = store.Mutations.OfContainer((c: any) => c.as_items((i: any) => i.mutations([
    store.Mutations.OfAny((a: any) => a.as_scalar((s: any) => s.mutations([
      store.Mutations.OfScalar((x: any) => x.set((v: any) => v.value(0n))),
    ]))),
    store.Mutations.OfAny((a: any) => a.clear(store.Mutations.OfAny.Clear((c: any) => c))),
  ])));
  const edits = [
    store.Mutations.OfContainer((c: any) => c.insert((i: any) => i.slice((p: any) => p.as_object((s: any) => s.index(0n).items(["a", "b"]))))),
    store.Mutations.OfContainer((c: any) => c.remove((r: any) => r.index(1n).count(2n))),
    store.Mutations.OfContainer((c: any) => c.append((a: any) => a.items(["z"]))),
    store.Mutations.OfContainer((c: any) => c.prepend((a: any) => a.items(["y"]))),
    store.Mutations.OfContainer((c: any) => c.clear(store.Mutations.OfContainer.Clear((c: any) => c))),
    items,
  ];
  const held = edits.map((edit) => plain(edit, M.OfContainer));
  assert(equal(held.map(keys), [["insert"], ["remove"], ["append"], ["prepend"], ["clear"], ["as_items"]]));
  assert(equal(held[1], map({ remove: { index: { int: 1n }, count: 2n } })) && equal(held[4], map({ clear: {} })));
  assert(equal(held[5].get("as_items").get("mutations")[1], map({ clear: {} })));
  const branch = store.Mutations.OfAny((a: any) => a.as_union((u: any) => u.mutations([
    store.Mutations.OfUnion.AsBranch((b: any) => b.name("circle").mutations([
      store.Mutations.OfProperty((p: any) => p.clear(store.Mutations.OfProperty.Clear((c: any) => c))),
    ])),
  ])));
  const part = store.Mutations.OfAny((a: any) => a.as_intersection((i: any) => i.mutations([
    store.Mutations.OfIntersection.AsPart((b: any) => b.name("stamp").mutations([])),
  ])));
  assert(equal(plain(branch, M.OfAny), map({ as_union: { mutations: [{ name: "circle", mutations: [{ clear: {} }] }] } })));
  assert(equal(plain(part, M.OfAny), map({ as_intersection: { mutations: [{ name: "stamp", mutations: [] }] } })));
}

## MUT-04 · Relations: entries inserted and removed with their links and properties; an adjacency cleared; entries in modulo order

In [ ]:
{
  const entry = (knower: string, known: string, since: bigint) => store.Mutations.OfEntry((e: any) => e.mutations([
    store.Mutations.OfEntry.Member((m: any) => m.as_link((l: any) => l.name("knower").mutations([
      store.Mutations.OfLink((k: any) => k.set((s: any) => s.ref(ref(knower)))),
    ]))),
    store.Mutations.OfEntry.Member((m: any) => m.as_link((l: any) => l.name("known").mutations([
      store.Mutations.OfLink((k: any) => k.set((s: any) => s.ref(ref(known)))),
    ]))),
    store.Mutations.OfEntry.Member((m: any) => m.as_property((p: any) => p.name("since").mutations([
      store.Mutations.OfProperty((v: any) => v.set((s: any) => s.value(since))),
    ]))),
  ]));

  const inserted = store.Mutations.Reference((u: any) => u.insert((i: any) => (
    i
      .relation(ref("Schemas.OfRelation/crm.Knows"))
      .entries([entry("crm.Person/ada", "crm.Person/bob", 2020n)])
  )));
  const removed = store.Mutations.Reference((u: any) => u.remove((r: any) => (
    r
      .relation(ref("Schemas.OfRelation/crm.Knows"))
      .entries([entry("crm.Person/ada", "crm.Person/bob", 2020n)])
  )));
  const cleared = store.Mutations.Reference((u: any) => u.clear((c: any) => c.ref(ref("crm.Person/ada")).adjacency("acquaintances")));
  const renumbered = store.Mutations.Reference((u: any) => u.as_entries((a: any) => (
    a
      .ref(ref("crm.Person/ada"))
      .adjacency("acquaintances")
      .mutations([store.Mutations.OfEntry.AsProperty((p: any) => p.name("since").mutations([
        store.Mutations.OfProperty((v: any) => v.set((s: any) => s.value(2021n))),
      ]))])
  )));
  const unlinked = store.Mutations.OfLink((k: any) => k.clear(store.Mutations.OfLink.Clear((c: any) => c)));
  const written = plain(log(inserted, removed, cleared, renumbered));
  assert(equal(written.map(keys), [["insert"], ["remove"], ["clear"], ["as_entries"]]));
  const first = written[0].get("insert").get("entries")[0].get("mutations");
  assert(equal(first[0], map({ as_link: { name: "knower", mutations: [{ set: { ref: { key: "crm.Person/ada" } } }] } })));
  assert(equal(first[2], map({ as_property: { name: "since", mutations: [{ set: { value: { int: 2020n } } }] } })));
  assert(equal(written[2], map({ clear: { ref: { key: "crm.Person/ada" }, adjacency: "acquaintances" } })));
  assert(equal(plain(unlinked, M.OfLink), map({ clear: {} })));
}

## MUT-05 · A log read back from plain data and from JSON, as any value is

In [ ]:
{
  const exists = store.Mutations.Reference((u: any) => u.exists((e: any) => e.definition((p: any) => p.as_object((d: any) => (
    d
      .key("crm.Person/ada")
      .properties({ name: "Ada" })
  )))));
  const written = plain(log(exists));
  const back = Plain.FromPlain(store)(M.Log, written);
  assert(equal(plain(back), written));
  const text = JSON.ToJSON(store)(M.Log, back);
  assert(equal(JSON.loads(text), written) && equal(plain(JSON.FromJSON(store)(M.Log, text)), written));
  raises(Error, () => Plain.FromPlain(store)(M.Log, [map({ exists: {}, clear: {} })])); // a union value holds one branch
}